# A swarm, from a cell

A *swarm* is a Diluvium program that spawns other Diluvium programs. Each
one is a real sandboxed instance with its own Lua state, its own memory
budget and its own capabilities — and a child can never hold more than its
parent did.

Everything below runs in this notebook. The **Instances** panel (the ⧉ on
the rail) shows the same swarm while it runs, drawn from the same data, so
open it alongside if you want to watch.

This needs a runtime that publishes `diluvium_swarm_wasi.wasm`, which
arrived in **v5.5.1_build5**. The next cell says whether yours does.


In [ ]:
-- The probe, not a version string: a build either has the layer or not.
if type(swarm) ~= "table" then
  print("This runtime publishes no swarm module.")
  print("Pick v5.5.1_build5 or newer in the Runtime dropdown, then run again.")
else
  local names = {}
  for k in pairs(swarm) do names[#names + 1] = k end
  table.sort(names)
  print("swarm is here. Calls available:")
  print("  " .. table.concat(names, ", "))
end


## The one thing that trips everybody up

A guest gets `inbox` and `outbox` for free (`doc/Messaging.md` §6.6).
It does **not** get `system/lifecycle` or `system/events` — those are
§9.2's reserved names, and a program that wants them **declares** them:

```lua
local sys = queue.declare("system/lifecycle", { capacity = 16 })  -- yes
local sys = queue.lookup("system/lifecycle")                      -- nil
```

`lookup` returns nil, and the failure surfaces much later as
`bad argument #1 to 'push' (number expected, got nil)` — pointing at the
push rather than at the declaration that never happened.

`lifecycle` is also a **capability**, and declaring the queue is not what
grants it. The capability set the parent hands down is.


In [ ]:
-- A root that spawns two workers and reports what it saw.
root_src = [[
local outbox = queue.lookup("outbox")
local sys    = queue.declare("system/lifecycle", { capacity = 8 })
local events = queue.declare("system/events", { capacity = 32 })

local worker = [==[
  local out = queue.lookup("outbox")
  queue.push(out, "worker " .. tostring(NAME) .. " ran")
]==]

for _, name in ipairs({ "alpha", "beta" }) do
  queue.push(sys, {
    op = "spawn",
    code = "NAME = " .. string.format("%q", name) .. "\n" .. worker,
    caps = { "queue:*" },
    budget = { instructions = 200000, memory_kb = 64 },
  })
end

local seen = 0
while seen < 2 do
  local _, e = queue.wait({ events })
  if e.event == "spawned" then
    seen = seen + 1
    queue.push(outbox, "spawned instance " .. tostring(e.id))
  end
end
]]

swarm.stop()
swarm.start{
  root = root_src,
  caps = { "lifecycle", "queue:*" },
  budget = { instructions = 5000000, memory_kb = 512 },
  max_instances = 8,
  spawns_per_step = 4,
}
print("started")


`swarm.start` builds the swarm and returns; nothing runs until you drive
it. `swarm.step(n)` takes up to *n* steps and hands back the events of that
window — a swarm that goes quiet stops early rather than spinning.


In [ ]:
local evs = swarm.step(20)
print(#evs .. " event(s):")
for _, e in ipairs(evs) do
  print(("  %-9s #%-3s %s"):format(e.event, tostring(e.id), tostring(e.detail or "")))
end

print()
print("what the root said:")
for _, line in ipairs(swarm.drain("root", "outbox")) do
  print("  " .. tostring(line))
end


## A grant may only narrow

This is the runtime's rule, not the host's, and it is worth seeing refused
rather than taking on trust. The root below holds `lifecycle` and asks for
it *for its child* — but the child's set is validated against what may be
passed down, and a request to widen is `denied` with the child never
created.

Granting exactly what you hold is the equality case and is allowed.
Granting more is not.


In [ ]:
swarm.stop()
swarm.start{
  root = [[
    local outbox = queue.lookup("outbox")
    local sys    = queue.declare("system/lifecycle", { capacity = 8 })
    local events = queue.declare("system/events", { capacity = 32 })
    queue.push(sys, { op = "spawn", code = "return",
                      caps = { "lifecycle", "queue:*", "host:sql/exec" } })
    local _, e = queue.wait({ events })
    queue.push(outbox, tostring(e.event) .. ": " .. tostring(e.detail))
  ]],
  caps = { "lifecycle", "queue:*" },
  budget = { instructions = 2000000, memory_kb = 256 },
  max_instances = 4,
}
swarm.step(10)
for _, line in ipairs(swarm.drain("root", "outbox")) do
  print(tostring(line))
end


## A budget stops a runaway

`run_lua` cannot be interrupted, so a program that loops forever is not
something you *stop* — it is something you **bound in advance**. Every
instance carries an instruction budget, and exceeding it ends that instance
and nothing else. The root below spawns a child that never returns.

Watch the word the child's death arrives under. The **guest** reads
`faulted` from `system/events`: its child died, and that is all a sibling
program is entitled to know. The **host** knows more — the Instances panel
shows the same death as `exceeded`, with the instruction count against the
budget that stopped it, because the host asked `dv_exceeded` and the guest
did not. Two views of one fact, and neither is the wrong one.


In [ ]:
swarm.stop()
swarm.start{
  root = [[
    local outbox = queue.lookup("outbox")
    local sys    = queue.declare("system/lifecycle", { capacity = 8 })
    local events = queue.declare("system/events", { capacity = 32 })
    queue.push(sys, { op = "spawn", code = "while true do end",
                      caps = { "queue:*" },
                      budget = { instructions = 300000, memory_kb = 32 } })
    while true do
      local _, e = queue.wait({ events })
      if e.event == "exceeded" or e.event == "faulted" or e.event == "exited" then
        queue.push(outbox, "child " .. tostring(e.id) .. " ended: " .. tostring(e.event))
        break
      end
    end
  ]],
  caps = { "lifecycle", "queue:*" },
  budget = { instructions = 50000000, memory_kb = 512 },
  max_instances = 4,
}
swarm.step(400)
for _, line in ipairs(swarm.drain("root", "outbox")) do
  print(tostring(line))
end
print("and the notebook is still here, which is the point.")


## The shape

`swarm.mermaid()` returns the topology as Mermaid text — the same graph the
Instances panel draws, from the same model, so the two cannot disagree
about what happened.

Three kinds of edge and they mean different things. **spawn** is exact:
`dvs_parent` said so at create time. **Queue edges** are routes the guest
declared, dotted until something has crossed them. There are no
instance-to-instance edges, because there is no such thing — a message
leaves on an exported queue, the host takes it, and the host decides what
happens next.

Paste the output into a PR, or use **View → Diagram renderer…** to have the
Lab draw it in the page.


In [ ]:
print(swarm.mermaid())


## Instances, as they were created

`events` renders the §9.2 event stream the way the panel does — the same
records a program reads from `system/events`, so what you see styled here
is what a supervisor actually branches on.


In [ ]:
swarm.stop()
swarm.start{
  root = root_src,
  caps = { "lifecycle", "queue:*" },
  budget = { instructions = 5000000, memory_kb = 512 },
  max_instances = 8,
}
events(swarm.step(20))


## Where this goes

Everything above is host-portable. `doc/Host.md`'s acceptance test is that
**a guest must not be able to tell two hosts apart** — so a supervisor
prototyped here is the same bytes that run against the C host, where the
connectors reach the system SQLite and a real socket instead of a database
in memory and a composer in a panel.

What differs is stated rather than hidden, in the panel and in the README:
the listener's port is recorded and never bound, and the SQL connector's
*confinement* is weaker than the C one's, because no JavaScript driver
exposes SQLite's authorizer. The contract is identical; build to it and
your program will not notice the move.

`swarm.stop()` frees the swarm and takes every instance's Lua state with
it. It is not an interrupt, which is why the panel's button says Stop.


In [ ]:
swarm.stop()
print("stopped. The roster in the panel is the last thing the host saw.")
